In [1]:
import pandas as pd
import ast
import re
import os

INPUT_FILE = "data/all_col_names.csv"
OUTPUT_FILE = "outputs/bitcoin_metrics_full_classification_final.csv"


# Convert the tokens column into a usable list
# If tokens is missing or malformed, fall back to splitting the column name
def safe_tokenize(row):
    name = str(row["column_name"]).lower()

    try:
        if "tokens" in row and isinstance(row["tokens"], str):
            tokens = ast.literal_eval(row["tokens"])
        elif "tokens" in row:
            tokens = row["tokens"]
        else:
            tokens = name.split("_")

        tokens = [str(t).lower() for t in tokens]

    except Exception:
        tokens = name.split("_")

    return tokens


In [2]:
# First meaningful token
def extract_root(tokens):
    return tokens[0] if len(tokens) > 0 else "none"


# Second meaningful token
def extract_sub_root(tokens):
    return tokens[1] if len(tokens) > 1 else "none"


# Extract time windows like 7d, 1m, 1y
def extract_time_horizon(tokens):
    time_pattern = re.compile(r"^_?\d+[dwmyh]$")
    horizons = [t.strip("_") for t in tokens if time_pattern.match(t)]
    return ",".join(horizons) if horizons else "spot"


# Extract units from the metric name
def extract_unit(tokens):
    units = [
        "usd", "btc", "sats", "sat",
        "cents", "percent", "pct"
    ]

    found = [t for t in tokens if t in units]
    return "_".join(found) if found else "none"


# Extract a broad metric type label from known keywords
def extract_metric_type(tokens):
    time_pattern = re.compile(r"^_?\d+[dwmyh]$")
    subject_tokens = [t for t in tokens if not time_pattern.match(t)]

    metric_keywords = [
        "price", "ratio", "mvrv", "sma", "ema",
        "returns", "return", "cap", "supply",
        "profit", "loss", "pnl", "cagr",
        "stack", "count", "dominance", "flow",
        "sopr", "nupl", "nvt", "sent",
        "received", "fee", "fees", "subsidy",
        "reward", "value", "basis", "cost",
        "difficulty", "hash", "hashrate", "volume",
        "drawdown", "velocity", "index",
        "rsi", "macd", "stoch", "puell",
        "reserve", "sortino", "downside",
        "coinbase", "blocks", "mined",
        "cumulative", "average", "median",
        "max", "min", "sum", "utxo"
    ]

    found = [t for t in subject_tokens if t in metric_keywords]
    return "_".join(found) if found else (subject_tokens[-1] if subject_tokens else "none")


# Assign each metric into a family
# Exact overrides are checked first, then broader rules
def assign_category(metric: str) -> str:
    m = str(metric).lower()

    profitability_sopr_exact = {
        "adjusted_sopr_30d_ema",
        "adjusted_sopr_7d_ema",
        "sopr_30d_ema",
        "sopr_7d_ema",
        "net_realized_pnl_7d_ema",
        "realized_profit_7d_ema",
        "realized_loss_7d_ema",
    }

    market_valuation_exact = {
        "investor_price_ratio_1m_sma",
        "investor_price_ratio_1w_sma",
        "investor_price_ratio_1y_sma",
        "investor_price_ratio_2y_sma",
        "investor_price_ratio_4y_sma",
        "investor_price_ratio_sma",
        "pi_cycle",
        "puell_multiple",
        "reserve_risk",
        "sell_side_risk_ratio_30d_ema",
        "sell_side_risk_ratio_7d_ema",
    }

    network_activity_exact = {
        "sent_14d_ema",
        "sent_14d_ema_btc",
        "sent_14d_ema_usd",
        "sent_in_loss_14d_ema",
        "sent_in_loss_14d_ema_btc",
        "sent_in_loss_14d_ema_usd",
        "sent_in_profit_14d_ema",
        "sent_in_profit_14d_ema_btc",
        "sent_in_profit_14d_ema_usd",
        "vocdd",
        "vocdd_365d_median",
        "vocdd_cumulative",
        "hash_rate_1m_sma",
        "hash_rate_1w_sma",
        "hash_rate_1y_sma",
        "hash_rate_2m_sma",
    }

    block_reward_exact = {
        "subsidy_usd_1y_sma",
    }

    technical_indicators_exact = {
        "macd_histogram",
        "macd_line",
        "macd_signal",
        "rsi_14d",
        "rsi_14d_max",
        "rsi_14d_min",
        "rsi_average_gain_14d",
        "rsi_average_loss_14d",
        "rsi_gains",
        "rsi_losses",
        "stoch_d",
        "stoch_k",
        "stoch_rsi",
        "stoch_rsi_d",
        "stoch_rsi_k",
        "sortino_1m",
        "sortino_1w",
        "sortino_1y",
        "downside_returns",
        "downside_1m_sd_sd",
        "downside_1m_sd_sma",
        "downside_1w_sd_sd",
        "downside_1w_sd_sma",
        "downside_1y_sd_sd",
        "downside_1y_sd_sma",
    }

    if m in profitability_sopr_exact:
        return "Profitability & SOPR"

    if m in market_valuation_exact:
        return "Market & Valuation"

    if m in network_activity_exact:
        return "Network Activity"

    if m in block_reward_exact:
        return "Block Reward Distributions"

    if m in technical_indicators_exact:
        return "Technical Indicators"

    if m in (
        "day_utc", "dateindex", "monthindex", "weekindex",
        "timestamp", "datetime"
    ):
        return "Metadata"

    if m.startswith("constant_"):
        return "Metadata"

    if m.startswith(("year_", "epoch_", "halvingepoch")):
        return "Age & Halving Cohorts"

    if m in ("days_before_next_halving", "blocks_before_next_halving"):
        return "Age & Halving Cohorts"

    if "pool" in m:
        return "Pool-Specific Economics"

    if "dominance" in m:
        return "Entities"

    if m.startswith("coinbase_usd_"):
        return "Market & Valuation"

    if m.startswith("coinbase_btc_"):
        return "Block Reward Distributions"

    if m.startswith("coinbase_"):
        return "Block Reward Distributions"

    if m.startswith("unclaimed_rewards"):
        return "Block Reward Distributions"

    if m.startswith("price_"):
        return "Price"

    if m.startswith(("utxo_", "utxos_")):
        return "UTXO Age Cohorts"

    if m in ("utxo_count", "exact_utxo_count"):
        return "UTXO Age Cohorts"

    if m.startswith(("sth_", "lth_")):
        return "Holder Cohorts"

    if m.startswith("addrs_"):
        return "Address Distribution by Balance"

    if m in ("total_addr_count",):
        return "Address Distribution by Balance"

    if m.startswith(("empty_outputs_", "emptyoutput_", "unknown_outputs_")):
        return "Network Activity"

    if m.startswith((
        "p2a_", "p2ms_", "p2pk_", "p2pkh_",
        "p2pk33_", "p2pk65_", "p2sh_",
        "p2wpkh_", "p2wsh_", "p2tr_",
        "p2sh_p2wpkh_", "p2sh_p2wsh_"
    )):
        return "Address Activity by Tech Type"

    if m.startswith((
        "dca_", "lump_sum_", "1d_", "1w_", "1m_", "3m_",
        "6m_", "1y_", "2y_", "3y_", "4y_", "5y_",
        "6y_", "8y_", "10y_", "_30d_", "30d_",
        "60d_", "90d_", "180d_", "365d_", "24h_"
    )):
        return "Benchmarks & DCA"

    if any(m.startswith(p) for p in (
        "rsi_", "macd_", "stoch_", "sortino_", "downside_"
    )):
        return "Technical Indicators"

    if any(k in m for k in (
        "_rsi", "_macd", "_stoch"
    )):
        return "Technical Indicators"

    if any(m.startswith(p) for p in (
        "sopr", "adjusted_sopr", "net_realized", "unrealized_",
        "pain_", "profit_", "capitulation_", "sell_side",
        "invested_capital", "neg_realized", "neg_unrealized",
        "nupl", "peak_regret", "loss_", "net_unrealized"
    )):
        return "Profitability & SOPR"

    if any(k in m for k in (
        "profit", "loss", "pnl", "sopr",
        "realized_profit", "realized_loss",
        "unrealized_profit", "unrealized_loss",
        "net_realized", "net_unrealized",
        "capitulation", "peak_regret"
    )):
        return "Profitability & SOPR"

    if any(m.startswith(p) for p in (
        "supply_", "subsidy_", "inflation_", "circulating_",
        "illiquid_", "liquid_", "highly_liquid_", "hodl_",
        "liveliness", "vaultedness", "cdd_", "coindays_",
        "coinblocks_", "cointime_", "thermocap_", "thermo_"
    )):
        return "Supply & Scarcity"

    if any(k in m for k in (
        "supply", "scarcity", "hodl", "liveliness",
        "vaultedness", "cointime", "coindays",
        "coinblocks", "coin_days", "cdd",
        "thermocap", "thermo_cap"
    )):
        return "Supply & Scarcity"

    if any(m.startswith(p) for p in (
        "block_", "blocks_", "tx_", "transaction_",
        "transactions_", "address_", "addr_", "fee_",
        "fees_", "hash_", "hashrate_", "hash_rate_",
        "sent_", "received_", "difficulty", "segwit_",
        "taproot_", "height_", "mempool_", "vsize_",
        "weight_", "input_", "inputs_", "output_",
        "outputs_", "empty_addr_", "new_addr_", "opreturn_"
    )):
        return "Network Activity"

    if any(k in m for k in (
        "block", "tx_", "transaction", "mempool",
        "hashrate", "hash_rate", "difficulty",
        "segwit", "taproot", "vsize", "weight",
        "input_count", "output_count", "outputs_per_sec",
        "inputs_per_sec", "opreturn", "empty_addr",
        "new_addr", "addr_count", "sent"
    )):
        return "Network Activity"

    if m in (
        "adjusted_value_created", "adjusted_value_destroyed",
        "value_created", "value_destroyed", "sent", "first_height"
    ):
        return "Network Activity"

    entity_metric_patterns = (
        "_blocks_mined", "_blocks_mined_cumulative",
        "_blocks_since_block", "_coinbase",
        "_days_since_block", "_fee", "_fee_btc",
        "_fee_usd", "_fee_cumulative",
        "_fee_btc_cumulative", "_fee_usd_cumulative",
        "_subsidy"
    )

    if any(p in m for p in entity_metric_patterns):
        return "Entities"

    entity_keywords = (
        "bitcoincom_", "btccom_", "foundry_", "foundryusa_",
        "slush_", "ghash_", "ghashio_", "btcguild_",
        "asicminer_", "antpool_", "f2pool_", "viabtc_",
        "binancepool_", "braiins_", "luxor_", "mara_",
        "riot_", "bitfury_", "bitfarms_", "nicehash_",
        "eclipsemc_", "eligius_", "btcc_", "btctop_",
        "cloudhashing_", "bitclub_", "bitminter_",
        "ozcoin_", "ocean_", "axbt_", "bcmonster_"
    )

    if m.startswith(entity_keywords):
        return "Entities"

    if any(m.startswith(p) for p in (
        "market_cap", "realized_cap", "mvrv", "nvt",
        "investor_", "cost_basis", "active_", "vaulted_",
        "true_market", "lower_price", "upper_price",
        "greed_", "oracle_", "terminal_", "balanced_",
        "delta_", "average_cap", "thermo_price",
        "max_cost_basis", "min_cost_basis",
        "btc_velocity", "gini"
    )):
        return "Market & Valuation"

    if any(k in m for k in (
        "price", "market_cap", "realized_cap", "mvrv", "nvt",
        "valuation", "cost_basis", "true_market",
        "terminal_price", "balanced_price", "delta_price",
        "average_cap", "velocity", "gini",
        "annualized_volume", "growth_rate", "cap_growth_rate",
        "net_sentiment", "greed_index", "pain_index",
        "realized_value", "spot_invested_capital_percentile"
    )):
        return "Market & Valuation"

    mining_keywords = (
        "miner_", "miners_", "mining_",
        "block_reward", "block_rewards",
        "hashrate_", "hash_rate", "subsidy_", "reward_"
    )

    if any(k in m for k in mining_keywords):
        return "Block Reward Distributions"

    return "Other"


# Build a simplified stem by removing time horizons and common suffixes
def create_stem(col):
    col = str(col).lower()

    col = re.sub(r"_(_?\d+[dwmyh])_", "_", col)
    col = re.sub(r"_(_?\d+[dwmyh])$", "", col)

    suffixes_to_remove = [
        "_cents", "_sats", "_sat", "_usd", "_btc",
        "_0sd_usd", "_sma", "_ema", "_ratio",
        "_zscore", "_z_score", "_normalized", "_standardized"
    ]

    changed = True

    while changed:
        changed = False
        for suffix in suffixes_to_remove:
            if col.endswith(suffix):
                col = col[:-len(suffix)]
                changed = True

    col = re.sub(r"__+", "_", col)
    col = col.strip("_")
    return col


In [3]:
# Parse one metric name into structured metadata fields
def parse_column(row):
    name = str(row["column_name"]).lower()
    tokens = safe_tokenize(row)

    time_pattern = re.compile(r"^_?\d+[dwmyh]$")
    subject_tokens = [t for t in tokens if not time_pattern.match(t)]

    root = extract_root(subject_tokens)
    sub_root = extract_sub_root(subject_tokens)

    metric_type = extract_metric_type(tokens)
    time_horizon = extract_time_horizon(tokens)
    unit = extract_unit(tokens)
    family = assign_category(name)
    stem = create_stem(name)

    # Cluster key groups similar columns together for later reduction
    cluster_key = (
        str(family)
        + " | root=" + str(root)
        + " | sub_root=" + str(sub_root)
        + " | type=" + str(metric_type)
        + " | time=" + str(time_horizon)
        + " | unit=" + str(unit)
    )

    return pd.Series([
        root, sub_root, metric_type, time_horizon, unit,
        family, stem, cluster_key
    ])

